# Evaluation: all paths in one comparison space

Scores every path's generated samples against the same held-out real audio, in the same space,
with the same metrics. Nothing is trained here.

**The rule this notebook follows:** never compare numbers computed in each path's own
representation. Every path ends as audio, so everything is judged there:

1. **Decode to waveform.** Path B (raw) and Path B-codec already produce audio. Paths A and C produce
   spectrograms with no vocoder, so they are inverted with Griffin-Lim.
2. **Representation ceilings.** Real held-out clips are pushed through each path's representation
   and straight back (no generator): mel -> Griffin-Lim, linear -> Griffin-Lim, codec encode ->
   decode, and mu-law (exact, so raw B's ceiling is the real audio itself). A ceiling is the best
   that path could ever score; the distance between ceiling and generated samples is what the
   generator itself loses.
3. **One fixed external embedding.** All audio (real, ceilings, generated) is embedded by models
   trained on neither our data nor any path's representation: PANNs CNN14 and LAION-CLAP. A
   codec-based embedder is deliberately not used; it would favour Path B-codec. Results are
   reported under both embedders, because Frechet distance's agreement with human judgement is known
   to depend on the embedding.
4. **Calibrated metrics.** `eval_metrics.evaluate`: Frechet distance split into mean and covariance
   terms, kernel MMD with a permutation test, and every number shown against a real-vs-real floor
   built from the same held-out data. Crops from the same source recording always stay on the same
   side of a split.

A secondary section repeats the comparison in mel space, the method used so far, for continuity
with earlier results and Ali's question about mel vs non-mel shift.

## 1. Setup

In [ ]:
!pip install -q diffusers accelerate soundfile panns_inference "transformers>=4.45"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, csv, json
REPO_DIR = "/content/thesis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/1jvaldez1/audio-diffusion-synth.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import common as C
import eval_metrics as M

import numpy as np
import matplotlib.pyplot as plt
import torch
import torchaudio
from pathlib import Path
from IPython.display import Audio, display

assert hasattr(C, "CodecFrontEnd") and hasattr(C, "spec_to_wave"), "common.py is an old version: pull the repo, then restart the session"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. Config

`PATHS` lists every sample file to score. A missing file is skipped with a note, so the notebook
also runs before every path has finished training.

In [ ]:
cfg = C.Config()
SEED = cfg.seed
S = f"{cfg.out_root}/samples"
EVAL_DIR = Path(f"{cfg.out_root}/eval")          # cached waveforms, embeddings, and results
EVAL_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {                                        # name -> (sample file, what the file holds)
    "A_mel":    (f"{S}/path_a.pt",            "mel"),
    "B_raw":    (f"{S}/path_b_wave.pt",       "waveform"),
    "B_codec":  (f"{S}/path_b_codec_wave.pt", "waveform"),          # diffusion over stacked token embeddings (negative result)
    "B_latent": (f"{S}/path_b_latent_wave_k64.pt", "waveform"),     # diffusion over DAC's continuous latent; match LATENT_DIM
    "C_linear": (f"{S}/path_c_linear.pt",     "linear"),
}
CEILING_OF = {}                                  # name -> path whose representation ceiling it shares
GL_ITERS  = 64                                   # Griffin-Lim iterations for Paths A and C
EMBEDDERS = ("panns", "clap")                    # the first one is the primary
MAX_DIM   = 32                                   # PCA dimensions kept for the embedding-space metrics
N_SPLITS, N_PERM = 30, 1000

## 3. Provenance check

Before any number is computed: are the runs comparable? This reads the settings stored with each
sample file and flags anything that differs from the current shared config or from the other runs:
training steps, data or diffusion settings, missing code versions, and runs made before the
validation fold existed (those trained on fold 4 as well, so they saw 400 more clips).

In [ ]:
COMPARE_KEYS = ["sample_rate", "n_fft", "hop_length", "n_mels", "spec_frames", "db_range", "val_fold",
                "heldout_fold", "crop_energy_frac", "num_train_timesteps", "beta_schedule",
                "prediction_type", "p_uncond", "batch_size", "lr", "total_steps", "seed"]
from dataclasses import asdict
current = asdict(cfg)

samples, problems = {}, {}
for name, (file, kind) in PATHS.items():
    if not os.path.exists(file):
        print(f"{name:<9} not found, skipped: {file}")
        continue
    s = C.load_samples(file)
    s["kind"], s["file"] = kind, file
    samples[name] = s
    issues = []
    m, c = s["meta"], s["config"]
    if "val_fold" not in c:
        issues.append("made before the validation fold existed: fold 4 was in its training data")
    for k in COMPARE_KEYS:
        if k in c and c[k] != current[k]:
            issues.append(f"{k} = {c[k]} (current config: {current[k]})")
    if not m.get("commit"):
        issues.append("no code version recorded: made before the GitHub repo, from the Drive copy of common.py")
    if m.get("guidance", 1.0) != 1.0:
        issues.append(f"guidance = {m['guidance']}; distribution metrics need 1.0")
    problems[name] = issues

print(f"\n{'path':<16}{'n':>5}{'train steps':>13}{'params':>13}{'commit':>9}  extra")
for name, s in samples.items():
    m = s["meta"]
    extra = {k: m[k] for k in ("compand", "n_codebooks", "latent_dim", "whiten", "cache_tokens", "attn_levels", "use_freq_channel") if k in m}
    print(f"{name:<16}{len(s['samples']):>5}{m.get('train_steps', '?'):>13}{m.get('n_params', 0):>13,}{str(m.get('commit', '-')):>9}  {extra}")

labels_ref = next(iter(samples.values()))["labels"]
for name, s in samples.items():
    if not torch.equal(s["labels"], labels_ref):
        problems[name].append("class labels differ from the other paths' (sizes or class balance differ)")

print()
for name, issues in problems.items():
    for msg in issues:
        print(f"WARNING {name}: {msg}")
if not any(problems.values()):
    print("All runs share the current config.")

## 4. Real reference set

ESC-50 fold 5 (never seen in training), two evenly spaced valid crops per clip: 400 clips give 800
crops, matching the 800 generated samples per path. Each crop carries the id of its source
recording (`src_file` in ESC-50's metadata), so crops cut from one recording never end up on both
sides of a real-vs-real split. Treating them as independent would make the floor too small and every
path look worse.

In [ ]:
data = C.load_esc50(cfg)
train_split, val_split, heldout_split = C.split_by_fold(data, cfg)
real_wave, real_labels, clip_idx = C.reference_crops(heldout_split, cfg, k=2, return_clip_index=True)

meta_csv = Path(cfg.data_root) / "ESC-50-master" / "meta" / "esc50.csv"
if meta_csv.exists():
    with open(meta_csv) as f:
        rows = list(csv.DictReader(f))
    src = np.array([r["src_file"] for r in rows])[data["folds"].numpy() == cfg.heldout_fold]
    groups = src[clip_idx.numpy()]
else:
    print("ESC-50 metadata not found (data loaded from cache); grouping by clip instead of by source recording")
    groups = clip_idx.numpy()
print(f"real reference: {len(real_wave)} crops from {len(np.unique(clip_idx))} clips, {len(np.unique(groups))} source recordings")

mel_fe = C.MelFrontEnd.load(f"{cfg.out_root}/frontend.json", cfg, device=device)
lin_file = f"{cfg.out_root}/frontend_linear.json"
lin_fe = (C.LinearFrontEnd.load(lin_file, cfg, device=device) if os.path.exists(lin_file)
          else C.LinearFrontEnd(cfg, device=device).fit(train_split["waveforms"]))
codec_runs = [s for k, s in samples.items() if k.startswith("B_")  and "n_codebooks" in s["meta"]]
codec_n = codec_runs[0]["meta"]["n_codebooks"] if codec_runs else 12
assert all(s["meta"]["n_codebooks"] == codec_n for s in codec_runs), "codec runs use different numbers of codebooks"
codec_fe = C.CodecFrontEnd(cfg, n_codebooks=codec_n, device=device)   # round trips need no statistics
if "B_latent" in samples:
    lm = samples["B_latent"]["meta"]
    latent_fe = C.LatentFrontEnd.load(f"{cfg.out_root}/frontend_latent_{codec_n}cb.pt", codec_fe,
                                      k=lm["latent_dim"], whiten=lm["whiten"])

## 5. Everything to waveform, plus the ceilings

Griffin-Lim results are cached in `EVAL_DIR` and recomputed only when a sample file changes.

In [ ]:
def cached(key, source_file, fn):
    """Load EVAL_DIR/key.pt unless the source file changed since it was written."""
    path = EVAL_DIR / f"{key}.pt"
    stamp = os.path.getmtime(source_file) if source_file else 0.0
    if path.exists():
        d = torch.load(path)
        if d["stamp"] == stamp:
            return d["value"]
    value = fn()
    torch.save({"value": value, "stamp": stamp}, path)
    return value

def to_waveform(s):
    x = s["samples"]
    if s["kind"] == "waveform":
        return x.float().clamp(-1, 1)
    fe = mel_fe if s["kind"] == "mel" else lin_fe
    return C.spec_to_wave(x, fe, n_iter=GL_ITERS, seed=SEED)

waves = {name: cached(f"wave_{name}", s["file"], lambda s=s: to_waveform(s)) for name, s in samples.items()}

# Ceilings: REAL held-out crops through each representation and back, no generator.
# Rows line up one-to-one with real_wave (they are "aligned" candidates in the metrics).
ceilings = {
    "A_mel":    lambda: C.spec_to_wave(torch.cat([mel_fe(real_wave[i:i + 64]).cpu() for i in range(0, len(real_wave), 64)]), mel_fe, n_iter=GL_ITERS, seed=SEED),
    "C_linear": lambda: C.spec_to_wave(torch.cat([lin_fe(real_wave[i:i + 64]).cpu() for i in range(0, len(real_wave), 64)]), lin_fe, n_iter=GL_ITERS, seed=SEED),
    "B_codec":  lambda: codec_fe.roundtrip(real_wave),
    "B_latent": lambda: latent_fe.roundtrip(real_wave),           # codec + PCA truncation
    "B_raw":    lambda: real_wave.clone(),       # continuous mu-law is exactly invertible: the ceiling is the real audio
}
ceil_key = lambda k: (f"ceil_{k}_{codec_n}cb" if k == "B_codec" else
                      f"ceil_{k}_{codec_n}cb_k{latent_fe.k}{'_white' if latent_fe.whiten else ''}" if k == "B_latent" else f"ceil_{k}")
ceil_waves = {f"ceil_{k}": cached(ceil_key(k), None, fn)
              for k, fn in ceilings.items() if k in samples or k in [CEILING_OF.get(n) for n in samples]}
for name, w in {**waves, **ceil_waves}.items():
    print(f"{name:<14}{tuple(w.shape)}")

### Listen

For two classes: the real crop, the same crop through each representation (ceilings), then one
generated sample per path. Ceilings tell you what each representation can sound like at best.

In [ ]:
for c in torch.unique(real_labels)[torch.randperm(len(torch.unique(real_labels)))[:2]].tolist():
    i = int(torch.nonzero(real_labels == c)[0])
    print(f"=== {data['class_names'][c]}: real")
    display(Audio(real_wave[i, 0].numpy(), rate=cfg.sample_rate))
    for name, w in ceil_waves.items():
        if name != "ceil_B_raw":
            print(f"{name} (same crop through the representation)")
            display(Audio(w[i, 0].numpy(), rate=cfg.sample_rate))
    for name, s in samples.items():
        j = int(torch.nonzero(s["labels"] == c)[0])
        print(f"{name} (generated)")
        display(Audio(waves[name][j, 0].numpy(), rate=cfg.sample_rate))

## 6. Level check

The earlier Frechet results were confounded by energy: raw Path B looked better before energy
normalisation and worse after. Loudness differences are worth seeing directly before reading any
distance. Clip RMS in dB relative to full scale, plus peak levels.

In [ ]:
def rms_db(w):
    return 20 * torch.log10(w.pow(2).mean(dim=(1, 2)).sqrt().clamp(min=1e-6))

sets = {"real": real_wave, **ceil_waves, **waves}
print(f"{'set':<16}{'median dBFS':>12}{'IQR':>16}{'peak >= 0.999':>15}{'median peak':>13}")
for name, w in sets.items():
    r, peaks = rms_db(w), w.abs().amax(dim=(1, 2))
    q1, q3 = r.quantile(0.25).item(), r.quantile(0.75).item()
    print(f"{name:<16}{r.median().item():>12.1f}{f'[{q1:.1f}, {q3:.1f}]':>16}{(peaks >= 0.999).float().mean().item():>15.1%}{peaks.median().item():>13.3f}")
print("Real crops come from clips peak-normalised to 0.95. Raw Path B and the codec clip at 1; Griffin-Lim output")
print("(Paths A and C) is left unclipped, so a median peak above 1 there means phase reconstruction overshoots.")

## 7. Embeddings

* **PANNs CNN14** (AudioSet tagger, 2048-d, runs at 32 kHz). Of the PANNs family the Wavegram-LogMel
  variant correlated best with human ratings of environmental audio (Tailleur et al., in the project
  files); `panns_inference` ships CNN14, which is the standard variant.
* **LAION-CLAP** (`laion/clap-htsat-unfused`, 512-d, runs at 48 kHz).

Audio is resampled from 16 kHz; nothing above 8 kHz exists in any set, real or generated, so the
upsampling treats every set the same. Embeddings are cached per set.

In [ ]:
def load_panns():
    from panns_inference import AudioTagging
    tagger = AudioTagging(checkpoint_path=None, device=device)
    resample = torchaudio.transforms.Resample(cfg.sample_rate, 32000)
    def embed(w):
        return tagger.inference(resample(w[:, 0]).numpy())[1]
    return embed

def load_clap():
    from transformers import ClapModel, ClapFeatureExtractor
    model = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device).eval()
    fx = ClapFeatureExtractor.from_pretrained("laion/clap-htsat-unfused")
    resample = torchaudio.transforms.Resample(cfg.sample_rate, 48000)
    @torch.no_grad()
    def embed(w):
        audio = [a.numpy() for a in resample(w[:, 0])]
        inputs = {k: v.to(device) for k, v in fx(audio, sampling_rate=48000, truncation="rand_trunc", padding="repeatpad", return_tensors="pt").items()}
        out = model.get_audio_features(**inputs)
        return (out.pooler_output if hasattr(out, "pooler_output") else out).cpu().numpy()
    return embed

LOADERS = {"panns": load_panns, "clap": load_clap}

def embed_all(embedder, sets, batch=32):
    fn, out = None, {}
    for name, w in sets.items():
        path = EVAL_DIR / f"emb_{embedder}_{name}.npy"
        src = samples[name]["file"] if name in samples else None
        stamp = os.path.getmtime(src) if src else 0.0
        if path.exists() and path.with_suffix(".stamp").exists() and float(path.with_suffix(".stamp").read_text()) == stamp:
            out[name] = np.load(path); continue
        fn = fn or LOADERS[embedder]()
        out[name] = np.concatenate([fn(w[i:i + batch]) for i in range(0, len(w), batch)]).astype(np.float64)
        np.save(path, out[name]); path.with_suffix(".stamp").write_text(str(stamp))
    return out

emb = {e: embed_all(e, sets) for e in EMBEDDERS}
for e, d in emb.items():
    print(e, {k: v.shape for k, v in d.items()})

## 8. Main result

For each embedder, every set is compared with the real reference through `M.evaluate`:

* **ratio** = Frechet distance (squared W2) divided by the real-vs-real floor at the same sample size.
  1 means indistinguishable from a second draw of real data; 3 means three times the floor.
* **above** = fraction of the 30 splits where the set was farther from real than real was from
  itself. About 0.5 for something real-like, 1.0 for a clear difference.
* **MMD p** = permutation test of "same distribution". With 800 samples, any generator will
  usually be detected; p is evidence of a difference, not a quality score.

The summary then splits each path's excess over the floor into what its representation costs
(ceiling) and what its generator adds on top (generated minus ceiling).

In [ ]:
results, summary = {}, []
ceil_names = list(ceil_waves)
for e in EMBEDDERS:
    d = emb[e]
    cands = {k: v for k, v in d.items() if k != "real"}
    res = M.evaluate(d["real"], cands, groups_real=groups, aligned=ceil_names, max_dim=MAX_DIM,
                     standardize=True, n_splits=N_SPLITS, n_perm=N_PERM, seed=SEED)
    results[e] = res
    print(f"\n==================== {e} ====================")
    print(M.format_report(res["report"]))
    rep = res["report"]
    for name in samples:
        g, c = rep[name]["w2_sq"], rep.get(f"ceil_{CEILING_OF.get(name, name)}", {}).get("w2_sq")
        excess = g["value"] - g["floor"]
        rep_share = (c["value"] - c["floor"]) / excess if (c and excess > 0) else float("nan")
        summary.append(dict(embedder=e, path=name,
                            ceiling_ratio=c["ratio"] if c else float("nan"), generated_ratio=g["ratio"],
                            generated_above=g["frac_above"], representation_share=rep_share,
                            mean_sq=rep[name]["mean_sq"]["value"], cov_sq=rep[name]["cov_sq"]["value"],
                            mmd_p=res["mmd"][name]["p_value"], off_subspace=res["off_subspace"][name]))

print(f"\n{'embedder':<8}{'path':<16}{'ceiling':>9}{'generated':>11}{'above':>7}{'repr. share':>13}{'mean / cov':>14}{'MMD p':>8}{'off-sub':>9}")
for r in summary:
    split = r["mean_sq"] / (r["mean_sq"] + r["cov_sq"]) if (r["mean_sq"] + r["cov_sq"]) > 0 else float("nan")
    print(f"{r['embedder']:<8}{r['path']:<16}{r['ceiling_ratio']:>9.2f}{r['generated_ratio']:>11.2f}{r['generated_above']:>7.2f}"
          f"{r['representation_share']:>13.0%}{f'{split:.0%} / {1 - split:.0%}':>14}{r['mmd_p']:>8.3f}{r['off_subspace']:>9.2f}")
print("\nceiling / generated: Frechet ratio to the real-vs-real floor (1 = real-like)")
print("repr. share: part of the generated set's excess over the floor already present in the ceiling")
print("mean / cov: how the generated set's Frechet distance divides between a location shift and a shape difference")

In [ ]:
fig, axes = plt.subplots(1, len(EMBEDDERS), figsize=(6 * len(EMBEDDERS), 3.6), squeeze=False)
for ax, e in zip(axes[0], EMBEDDERS):
    rows = [r for r in summary if r["embedder"] == e]
    x = np.arange(len(rows))
    ax.bar(x - 0.2, [r["ceiling_ratio"] for r in rows], 0.4, label="ceiling (representation only)", color="#9aa5b1")
    ax.bar(x + 0.2, [r["generated_ratio"] for r in rows], 0.4, label="generated", color="#3b6ea8")
    ax.axhline(1.0, color="black", lw=1, ls="--")
    ax.set_xticks(x, [r["path"] for r in rows]); ax.set_yscale("log")
    ax.set_ylabel("Frechet distance / real-vs-real floor"); ax.set_title(f"{e} embedding")
axes[0][0].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Where each path falls short

The stretch spectrum (primary embedder) splits each path's covariance term into independent
directions. Below 1: the generator produces too little variation along that direction (collapse
toward common modes). Above 1: too much (noise, artefacts). The grey curve is real data against
itself, which shows how much spread sampling noise alone produces.

In [ ]:
e = EMBEDDERS[0]
Zr = results[e]["reducer"].transform(emb[e]["real"])
ug = np.unique(groups)
half = np.isin(groups, np.random.default_rng(SEED).permutation(ug)[: len(ug) // 2])
ref = M.stretch_spectrum(Zr[half], Zr[~half])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(np.sort(ref.stretch), color="gray", lw=2, label="real vs real")
for name in samples:
    s = results[e]["spectra"][name]
    ax.plot(np.sort(s.stretch), marker="o", ms=3, label=name)
    print(f"{name:<16} contraction {s.contraction_share:.0%} | expansion {s.expansion_share:.0%} | log volume ratio {s.log_volume_ratio:+.2f}")
ax.axhline(1.0, color="gray", lw=1, ls="--"); ax.set_yscale("log")
ax.set_xlabel("direction (sorted by stretch)"); ax.set_ylabel("stretch (1 = matches real)")
ax.set_title(f"{e}: below 1 too little variation, above 1 too much"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 9. Secondary: the same comparison in mel space

This is the method used so far: flatten the log-mel spectrogram and compare. It is kept for
continuity with earlier numbers and for Ali's question about distribution shift between mel and
non-mel representations. **Read it with its bias in mind:** mel is Path A's own representation, so
Path A skips Griffin-Lim here and its ceiling is zero by construction, while every other path pays
for a conversion. That asymmetry is why the embedding comparison above is the primary result.

Conversions: Path A as generated; both Path B variants through the shared mel front end; Path C
through `C.linear_to_mel`, with its ceiling (real audio through the same conversion) computed the
same way.

In [ ]:
to_mel = lambda w: torch.cat([mel_fe(w[i:i + 64]).cpu() for i in range(0, len(w), 64)])
real_mel = to_mel(real_wave)
mel_sets, mel_aligned = {}, []
for name, s in samples.items():
    if s["kind"] == "mel":
        mel_sets[name] = s["samples"]
    elif s["kind"] == "linear":
        mel_sets[name] = C.linear_to_mel(s["samples"].to(device), lin_fe, mel_fe).cpu()
        mel_sets["ceil_C_linear"] = C.linear_to_mel(torch.cat([lin_fe(real_wave[i:i + 64]) for i in range(0, len(real_wave), 64)]), lin_fe, mel_fe).cpu()
        mel_aligned.append("ceil_C_linear")
    else:
        mel_sets[name] = to_mel(waves[name])
for cname in ("ceil_B_codec", "ceil_B_latent"):
    if cname in ceil_waves:
        mel_sets[cname] = to_mel(ceil_waves[cname]); mel_aligned.append(cname)

flat = lambda t: t.reshape(len(t), -1).double().numpy()
res_mel = M.evaluate(flat(real_mel), {k: flat(v) for k, v in mel_sets.items()}, groups_real=groups,
                     aligned=mel_aligned, max_dim=20, standardize=False, n_splits=N_SPLITS, n_perm=N_PERM, seed=SEED)
results["mel"] = res_mel
print(M.format_report(res_mel["report"]))

## 10. Save

Writes the summary table and the full reports to `EVAL_DIR`, tagged with the code version, so a
result can always be traced to the exact runs and code that produced it.

In [ ]:
import subprocess
commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip()
out = {
    "commit": commit,
    "runs": {k: {"file": s["file"], "meta": {kk: (list(v) if isinstance(v, tuple) else v) for kk, v in s["meta"].items()}}
             for k, s in samples.items()},
    "warnings": problems,
    "summary": summary,
    "reports": {k: r["report"] for k, r in results.items()},
}
with open(EVAL_DIR / f"results_{commit}.json", "w") as f:
    json.dump(out, f, indent=2, default=float)
print("saved", EVAL_DIR / f"results_{commit}.json")

## How to read the result

* **A path's ceiling sets its limit.** If a ceiling ratio is already high, that representation loses
  too much for this data no matter how good the generator gets. Better training will not fix it.
* **The gap between ceiling and generated is the generator's job.** Comparing gaps across paths at
  equal parameters and equal training steps is the fair comparison of how learnable each
  representation is.
* **The two embedders should agree.** If PANNs and CLAP rank the paths differently, the honest
  statement is that the ranking is embedding-dependent, and it should be reported that way.
* **This is not yet the thesis question.** Distribution distance is a validity gate. Whether the
  synthetic audio is *close enough to train on* is answered by training a classifier on synthetic
  audio and testing on real audio (TSTR), which is the natural next notebook.